# 03 · Feature engineering

**Purpose.** Turn the cleaned daily series into a supervised dataset for **next-day** AQI
forecasting: lagged AQI, rolling statistics of past AQI, yesterday's pollutants, calendar
features and a city one-hot encoding. Two variants are built:

- **forecast**: every feature for day *t* uses only information available by the end of day
  *t-1*. This is the real model.
- **nowcast (leaky)**: same features plus *same-day* pollutant concentrations. It is used only in
  notebook 04 to show how much these inputs inflate the score.

**Inputs**
- `../data/processed/clean.csv` (from notebook 01)

**Outputs**
- `../data/processed/features_forecast.csv`
- `../data/processed/features_nowcast.csv`

## Configuration
Same values as the original `src/config.py`.

In [1]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

POLLUTANTS = ["PM2.5", "PM10", "NO2", "SO2", "CO", "O3", "NH3"]
TARGET = "AQI"
LAGS = [1, 2, 3, 7]
ROLL_WINDOWS = [3, 7]

In [2]:
import pandas as pd

## 1. Load the cleaned data
`float_precision="round_trip"` reads floats back bit for bit, so the features match the ones
computed in memory from notebook 01's output.

In [3]:
clean_df = pd.read_csv(PROCESSED_DIR / "clean.csv", parse_dates=["Date"], float_precision="round_trip")
print(clean_df.shape)
clean_df.head()

(14060, 11)


,City,Date,PM2.5,PM10,NO2,SO2,CO,O3,NH3,AQI,AQI_observed
0,Ahmedabad,2015-01-01,NaN,NaN,18.22,27.64,0.92,133.36,NaN,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,15.69,24.55,0.97,34.06,NaN,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,19.30,29.07,17.40,30.70,NaN,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,18.48,18.59,1.70,36.08,NaN,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,21.42,39.33,22.10,39.31,NaN,NaN,NaN


## 2. Feature builder
**Only past information (no leakage).** AQI is computed from same-day pollutant concentrations.
Same-day pollutants as inputs would let the model reverse-engineer the AQI formula, which is not a
forecast. So every feature for day *t* is shifted to end at *t-1*:

- `AQI_lag{1,2,3,7}`: AQI 1, 2, 3 and 7 days earlier
- `AQI_rollmean{3,7}`, `AQI_rollstd7`: rolling mean and std of AQI over the previous 3 or 7 days.
  The series is shifted by 1 *before* rolling, so the window never includes day *t*.
- `{pollutant}_lag1`: yesterday's pollutant concentrations

**Built inside each city.** Shifts and rolling windows run on each city's own series. Since notebook
01 made every city a continuous daily calendar, a shift of 1 is exactly one day, and the first rows
of one city never pick up the last values of the previous city.

**Calendar and city features** (`month`, `dayofweek`, `dayofyear`, `city_*` one-hot) are known in
advance, so they do not leak.

`include_same_day_pollutants=True` adds `{pollutant}_today` columns for the leaky nowcast variant.

In [4]:
def build_features(df: pd.DataFrame, include_same_day_pollutants: bool = False) -> pd.DataFrame:
    frames = []
    for _, g in df.groupby("City"):
        g = g.sort_values("Date").copy()
        for lag in LAGS:
            g[f"AQI_lag{lag}"] = g[TARGET].shift(lag)
        for w in ROLL_WINDOWS:
            g[f"AQI_rollmean{w}"] = g[TARGET].shift(1).rolling(w).mean()
        g["AQI_rollstd7"] = g[TARGET].shift(1).rolling(7).std()
        for pol in POLLUTANTS:
            g[f"{pol}_lag1"] = g[pol].shift(1)
        if include_same_day_pollutants:
            for pol in POLLUTANTS:
                g[f"{pol}_today"] = g[pol]
        frames.append(g)

    out = pd.concat(frames, ignore_index=True)
    out["month"] = out["Date"].dt.month
    out["dayofweek"] = out["Date"].dt.dayofweek
    out["dayofyear"] = out["Date"].dt.dayofyear
    city_names = out["City"].copy()
    out = pd.get_dummies(out, columns=["City"], prefix="city", dtype=int)
    out["City"] = city_names
    return out

Model inputs are all columns except identifiers, the target, the unfilled target and the raw
same-day pollutant columns.

In [5]:
def feature_columns(df: pd.DataFrame) -> list:
    exclude = {"Date", "City", TARGET, "AQI_observed"} | set(POLLUTANTS)
    return [c for c in df.columns if c not in exclude]

## 3. Forecast variant (information up to *t-1* only)

In [6]:
feats = build_features(clean_df)
cols = feature_columns(feats)
print(f"{feats.shape}   features={len(cols)}")
cols

(14060, 35)   features=24


['AQI_lag1',
 'AQI_lag2',
 'AQI_lag3',
 'AQI_lag7',
 'AQI_rollmean3',
 'AQI_rollmean7',
 'AQI_rollstd7',
 'PM2.5_lag1',
 'PM10_lag1',
 'NO2_lag1',
 'SO2_lag1',
 'CO_lag1',
 'O3_lag1',
 'NH3_lag1',
 'month',
 'dayofweek',
 'dayofyear',
 'city_Ahmedabad',
 'city_Bengaluru',
 'city_Chennai',
 'city_Delhi',
 'city_Hyderabad',
 'city_Lucknow',
 'city_Mumbai']

Check on Delhi's first days: `AQI_lag1` on each day equals the previous day's `AQI`, and
`AQI_rollmean3` averages the three days *before* the current one.

In [7]:
feats.loc[feats["City"] == "Delhi", ["Date", "AQI", "AQI_lag1", "AQI_lag2", "AQI_rollmean3", "PM2.5", "PM2.5_lag1"]].head(8)

,Date,AQI,AQI_lag1,AQI_lag2,AQI_rollmean3,PM2.5,PM2.5_lag1
6027,2015-01-01,472.0,NaN,NaN,NaN,313.22,NaN
6028,2015-01-02,454.0,472.0,NaN,NaN,186.18,313.22
6029,2015-01-03,143.0,454.0,472.0,NaN,87.18,186.18
6030,2015-01-04,319.0,143.0,454.0,356.333333,151.84,87.18
6031,2015-01-05,325.0,319.0,143.0,305.333333,146.60,151.84
6032,2015-01-06,318.0,325.0,319.0,262.333333,149.58,146.60
6033,2015-01-07,353.0,318.0,325.0,320.666667,217.87,149.58
6034,2015-01-08,383.0,353.0,318.0,332.000000,229.90,217.87


## 4. Nowcast variant (adds same-day pollutants, leaky)

In [8]:
leaky = build_features(clean_df, include_same_day_pollutants=True)
lcols = feature_columns(leaky)
print(f"{leaky.shape}   features={len(lcols)}")
[c for c in lcols if c not in cols]

(14060, 42)   features=31


['PM2.5_today',
 'PM10_today',
 'NO2_today',
 'SO2_today',
 'CO_today',
 'O3_today',
 'NH3_today']

## 5. Save both variants

In [9]:
feats.to_csv(PROCESSED_DIR / "features_forecast.csv", index=False)
leaky.to_csv(PROCESSED_DIR / "features_nowcast.csv", index=False)
print("Saved features_forecast.csv", feats.shape, "and features_nowcast.csv", leaky.shape)

Saved features_forecast.csv (14060, 35) and features_nowcast.csv (14060, 42)
